# import libraries

In [1]:
import json
import os
import subprocess
import time
from pathlib import Path
from dotenv import load_dotenv
import pandas as pd

# variables

In [2]:
load_dotenv()  
ENV_PATH = os.getenv("ENV_PATH")

In [5]:
from pathlib import Path

# ── Project root (notebook lives inside tessera_embeddings/) ──────────────────
PROJECT_ROOT = Path.cwd().parent

# ── Paths ─────────────────────────────────────────────────────────────────────
ROI_DIR      = (PROJECT_ROOT / "_data/exports/roi_tiffs").resolve()
DOWNLOAD_DIR = (PROJECT_ROOT / "_data/exports/s1_s2_downloads").resolve()
TEMP_DIR     = (PROJECT_ROOT / "_data/exports/temp").resolve()
TESSERA_DIR  = (PROJECT_ROOT / "tessera_embeddings").resolve()

# ── Python binary — project venv ──────────────────────────────────────────────
PYTHON_ENV = (Path(ENV_PATH))#.resolve()

# ── Download settings ─────────────────────────────────────────────────────────
DATA_SOURCE   = "mpc"
RESOLUTION    = 10.0
WORKERS       = 4    # Dask workers for both S1 and S2 processors
WORKER_MEMORY = 4    # GB per worker

# ── Create output directories ─────────────────────────────────────────────────
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)

# ── Verify everything is in place ─────────────────────────────────────────────
assert ROI_DIR.exists(),    f"ROI dir not found: {ROI_DIR}"
assert TESSERA_DIR.exists(), f"Tessera dir not found: {TESSERA_DIR}"
assert PYTHON_ENV.exists(),  f"Python env not found: {PYTHON_ENV}"

scripts = [
    "tessera_preprocessing/s1_fast_processor.py",
    "tessera_preprocessing/s2_fast_processor.py",
    "tessera_preprocessing/s1_s2_stacker.sh",
    "tessera_preprocessing/dpixel_retiler.py",
]
for s in scripts:
    assert (TESSERA_DIR / s).exists(), f"Missing: {TESSERA_DIR / s}"

print(f"ROI tiffs    : {ROI_DIR}")
print(f"Download dir : {DOWNLOAD_DIR}")
print(f"Tessera dir  : {TESSERA_DIR}")
print(f"Python env   : {PYTHON_ENV}")
print(f"Data source  : {DATA_SOURCE}")
print(f"Workers      : {WORKERS} × {WORKER_MEMORY}GB RAM each")
print("All paths verified ✓")

ROI tiffs    : /Users/devseed/Documents/docs/THESIS/SpatialTemporal_generalization_of_SITS_FM/_data/exports/roi_tiffs
Download dir : /Users/devseed/Documents/docs/THESIS/SpatialTemporal_generalization_of_SITS_FM/_data/exports/s1_s2_downloads
Tessera dir  : /Users/devseed/Documents/docs/THESIS/SpatialTemporal_generalization_of_SITS_FM/tessera_embeddings
Python env   : /Users/devseed/Documents/docs/THESIS/SpatialTemporal_generalization_of_SITS_FM/.venv/bin/python
Data source  : mpc
Workers      : 4 × 4GB RAM each
All paths verified ✓


# collect all points to download

In [7]:
meta_files = sorted(ROI_DIR.rglob("meta.json"))
print(f"Found {len(meta_files)} points to download")

points = []
for meta_path in meta_files:
    meta = json.loads(meta_path.read_text())
    meta["roi_tiff"]   = str(meta_path.parent / "roi.tiff")
    meta["meta_path"]  = str(meta_path)
    points.append(meta)

df_points = pd.DataFrame(points)

print(f"\nBreakdown by year:")
print(df_points.groupby("year").size().to_string())
print(f"\nBreakdown by country:")
print(df_points.groupby("country_id").size().to_string())

Found 43728 points to download

Breakdown by year:
year
2016     9669
2017     4669
2018    14698
2019    14692

Breakdown by country:
country_id
at    7500
be    7500
bg    7500
dk    7500
ee    7169
ie    6559


# download one point

In [8]:
def download_point(roi_tiff, year, out_dir, temp_dir, tessera_dir, python_env, 
                   data_source="mpc", resolution=10.0, workers=4, worker_memory=4):
    
    preproc_dir = tessera_dir / "tessera_preprocessing"
    out_dir.mkdir(parents=True, exist_ok=True)
    temp_dir.mkdir(parents=True, exist_ok=True)
    
    start_date = f"{year}-01-01T00:00:00"
    end_date   = f"{year}-12-31T23:59:59"
    
    s2_out = out_dir / "data_raw"
    s1_out = out_dir / "data_sar_raw"
    
    # ── S2 ────────────────────────────────────────────────────────────────────
    s2_result = subprocess.run([
        str(python_env), "s2_fast_processor.py",
        "--input_tiff",   str(roi_tiff),
        "--start_date",   start_date,
        "--end_date",     end_date,
        "--output",       str(s2_out),
        "--max_cloud",    "100",          # tessera default
        "--data_source",  data_source,
        "--dask_workers", str(workers),
        "--worker_memory", str(worker_memory),
        "--resolution",   str(resolution),
        "--min_coverage", "0.01",         # tessera default
        "--partition_id", "p0",
        "--overwrite",
    ],
        cwd=str(preproc_dir),
        capture_output=True,
        text=True,
    )
    
    if s2_result.returncode not in (0, 2):  # 2 = partial success, still ok
        print(f"\n  S2 STDOUT: {s2_result.stdout[-500:]}")
        print(f"  S2 STDERR: {s2_result.stderr[-500:]}")
        return False
    
    # ── S1 ────────────────────────────────────────────────────────────────────
    s1_result = subprocess.run([
        str(python_env), "s1_fast_processor.py",
        "--input_tiff",   str(roi_tiff),
        "--start_date",   start_date,
        "--end_date",     end_date,
        "--output",       str(s1_out),
        "--data_source",  data_source,
        "--dask_workers", str(workers),
        "--worker_memory", str(worker_memory),
        "--resolution",   str(resolution),
        "--orbit_state",  "both",         # asc + desc, tessera default
        "--min_coverage", "0.01",
        "--partition_id", "p0",
        "--overwrite",
    ],
        cwd=str(preproc_dir),
        capture_output=True,
        text=True,
    )
    
    if s1_result.returncode not in (0, 2):
        print(f"\n  S1 STDOUT: {s1_result.stdout[-500:]}")
        print(f"  S1 STDERR: {s1_result.stderr[-500:]}")
        return False
    
    return True


def is_already_downloaded(out_dir: Path) -> bool:
    """Check if download outputs already exist for this point."""
    s2_raw = out_dir / "data_raw"
    s1_raw = out_dir / "data_sar_raw"
    return (
        s2_raw.exists() and any(s2_raw.rglob("*.tif")) and
        s1_raw.exists() and any(s1_raw.rglob("*.tif"))
    )


print("Helper functions defined.")

Helper functions defined.


In [9]:
results = []
n_total    = len(points)
n_success  = 0
n_skipped  = 0
n_failed   = 0

for i, pt in enumerate(points):
    pt_label = f"{pt['file_id']}/point_{pt['row_index']:04d}"
    pt_out   = DOWNLOAD_DIR / pt["file_id"] / f"point_{pt['row_index']:04d}"
    pt_tmp   = TEMP_DIR     / pt["file_id"] / f"point_{pt['row_index']:04d}"

    print(f"[{i+1}/{n_total}] {pt_label}  (year={pt['year']}) ", end="", flush=True)

    # Skip if already done
    if is_already_downloaded(pt_out):
        print("→ already downloaded, skipping")
        n_skipped += 1
        results.append({**pt, "status": "skipped"})
        continue

    t0 = time.time()
    ok = download_point(
        roi_tiff     = pt["roi_tiff"],
        year         = pt["year"],
        out_dir      = pt_out,
        temp_dir     = pt_tmp,
        tessera_dir  = TESSERA_DIR,
        python_env   = PYTHON_ENV,
        data_source  = DATA_SOURCE,
        resolution   = RESOLUTION,
        workers      = WORKERS,      # replaces s1_partitions + s1_workers + s2_partitions + s2_workers
        worker_memory = WORKER_MEMORY,
    )
    elapsed = time.time() - t0

    if ok:
        print(f"→ ✓  ({elapsed:.1f}s)")
        n_success += 1
        results.append({**pt, "status": "success", "elapsed_s": elapsed})
    else:
        print(f"→ ✗ FAILED  ({elapsed:.1f}s)")
        n_failed += 1
        results.append({**pt, "status": "failed",  "elapsed_s": elapsed})

print(f"\n{'='*50}")
print(f"Success  : {n_success}")
print(f"Skipped  : {n_skipped}")
print(f"Failed   : {n_failed}")
print(f"Total    : {n_total}")

[1/43728] at_2017/point_0000  (year=2017) 

KeyboardInterrupt: 